# 评测数据整理与验收

数据源：_实验系统/telemetry/benchmark.sqlite

> 状态说明：本 Notebook 保留执行当时的输出快照；输出中的 HOLD、status_unverified 等词反映采集/整理时状态。当前正式评分状态以 SQLite 的 enhanced_review_1.0 与主工作簿为准。


In [1]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd

ROOT = Path(r"D:\项目2")
sys.path.insert(0, str(ROOT))
DB = ROOT / "_实验系统" / "telemetry" / "benchmark.sqlite"

# 只读连接
con = sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True)


## 1. 运行记录


In [2]:
# 标准化运行记录
run_table = pd.read_sql_query(
    """
    SELECT
        run_id, ai_name, condition_name, host_tool, model_name,
        status, started_at, ended_at
    FROM benchmark_run
    ORDER BY started_at
    """,
    con,
)

# 排除采集校准
run_table = run_table.loc[
    run_table["condition_name"].ne("calibration")
].copy()

# 轮次标签
condition_labels = {
    "baseline": "第一轮",
    "evidence_led_v1": "第二轮",
    "evidence_led_v1_supplemental_recheck": "第二轮补充",
}

run_view = (
    run_table
    .assign(轮次=run_table["condition_name"].map(condition_labels))
    .rename(columns={
        "ai_name": "AI",
        "host_tool": "宿主",
        "model_name": "模型",
        "status": "状态",
        "started_at": "开始时间",
        "ended_at": "结束时间",
    })
    [["AI", "轮次", "宿主", "模型", "状态", "开始时间", "结束时间"]]
    .reset_index(drop=True)
)

run_view


AI,轮次,宿主,模型,状态,开始时间,结束时间
GPT,第一轮,codex,gpt-6-astra,completed,2026-09-16T02:58:30.614392+00:00,2026-09-16T03:15:04.183793+00:00
Grok,第一轮,cursor,grok-4.6,completed,2026-09-16T05:36:15.802000+00:00,2026-09-16T05:46:22.243000+00:00
DeepSeek,第一轮,claude_code,deepseek-v4.1-flash,completed,2026-09-16T07:30:15.009000+00:00,2026-09-16T08:02:40.743000+00:00
Gemini,第一轮,antigravity,gemini-3.8-flash,completed,2026-09-16T07:37:01+00:00,2026-09-16T07:52:21+00:00
Qwen,第一轮,claude_code,qwen3.8-max-0902,completed,2026-09-16T11:26:39.828Z,2026-09-16T11:40:59.025Z
GLM,第一轮,claude_code,glm-5.3,completed,2026-09-16T13:09:00.990Z,2026-09-16T13:14:33.944Z
DeepSeek,第二轮,claude_code,deepseek-v4.1-flash,completed,2026-09-17T02:31:01.604247+00:00,2026-09-17T02:52:52.448618+00:00
GPT,第二轮,codex,gpt-6-astra,completed,2026-09-17T02:45:22.496837+00:00,2026-09-17T03:03:23.163838+00:00
Qwen,第二轮,claude_code,qwen3.8-max-0902,completed,2026-09-17T02:49:19.122103+00:00,2026-09-17T03:49:36.258352+00:00
GLM,第二轮,claude_code,glm-5.3,completed,2026-09-17T02:49:19.382178+00:00,2026-09-17T02:58:59.295935+00:00


## 2. 参与登记


In [3]:
# 实验登记
registry_table = pd.read_sql_query(
    """
    SELECT
        participation_id, ai, round_no, run_id,
        record_status, reason
    FROM evaluation_participation
    ORDER BY participation_id
    """,
    con,
)

# 按 run_id 对齐实际运行
registry_with_run = registry_table.loc[
    registry_table["run_id"].notna()
].copy()

run_registry = run_table.merge(
    registry_with_run,
    on="run_id",
    how="left",
    validate="one_to_one",
)

# Claude 第一轮混合成果
mixed_registry = registry_table.loc[
    registry_table["run_id"].isna()
    & registry_table["record_status"].eq("mixed_excluded")
].copy()

analysis_registry = pd.concat(
    [
        run_registry[
            [
                "participation_id", "ai", "round_no", "run_id",
                "record_status", "reason"
            ]
        ],
        mixed_registry[
            [
                "participation_id", "ai", "round_no", "run_id",
                "record_status", "reason"
            ]
        ],
    ],
    ignore_index=True,
)

registry_view = (
    analysis_registry
    .assign(轮次=analysis_registry["round_no"].map({1: "第一轮", 2: "第二轮"}))
    .rename(columns={
        "ai": "AI",
        "run_id": "运行编号",
        "record_status": "记录状态",
        "reason": "说明",
    })
    [["AI", "轮次", "运行编号", "记录状态", "说明"]]
)

registry_view


AI,轮次,运行编号,记录状态,说明
GPT,第一轮,20260916_105830_GPT_886636b7,completed,Existing baseline v1.0.
Grok,第一轮,20260916_133206_Grok_2a33e3a3,completed,Existing baseline v1.0.
DeepSeek,第一轮,20260916_153015_DeepSeek_a32f78bc,completed,Existing baseline v1.0.
Gemini,第一轮,20260916_153701_Gemini_1dd2676d,completed,Existing baseline v1.0.
Qwen,第一轮,20260916_192639_Qwen_8fc12d57,completed,Existing baseline v1.0.
GLM,第一轮,20260916_210900_GLM_ce9faeda,completed,Existing baseline v1.0.
DeepSeek,第二轮,20260917_103101_DeepSeek_6a24da15,completed,Completed observation; release remains HOLD.
GPT,第二轮,20260917_104522_GPT_b97ef10b,completed,Completed observation; release remains HOLD.
Qwen,第二轮,20260917_104919_Qwen_ca834dcf,completed,Completed observation; release remains HOLD.
GLM,第二轮,20260917_104919_GLM_b5270231,completed,Completed observation; release remains HOLD.


## 3. 评分与用量


In [4]:
# 原始冻结评分版本（本次裁定见下方配对核验）
score_table = pd.read_sql_query(
    """
    SELECT version, ai, criterion, score, maximum
    FROM evaluation_score_item
    """,
    con,
)

score_stage = {
    "1.0": "第一轮",
    "enhanced_review_1.0": "第二轮",
    "claude_assisted_quality_20260918_v1": "混合成果",
}

formal_scores = (
    score_table
    .loc[score_table["version"].isin(score_stage)]
    .assign(
        阶段=lambda x: x["version"].map(score_stage),
        评分维度=lambda x: x["criterion"].str.startswith("R").map({
            True: "独立可靠性",
            False: "成果质量",
        }),
    )
)

# A-E：成果质量；R1-R5：独立可靠性
score_detail = (
    formal_scores
    .groupby(["阶段", "ai", "评分维度"], as_index=False)
    .agg(
        得分=("score", "sum"),
        满分=("maximum", "sum"),
    )
)

score_value = (
    score_detail
    .pivot(index=["阶段", "ai"], columns="评分维度", values="得分")
    .reset_index()
)

score_maximum = (
    score_detail
    .pivot(index=["阶段", "ai"], columns="评分维度", values="满分")
    .reset_index()
    .rename(columns={
        "成果质量": "质量满分",
        "独立可靠性": "可靠性满分",
    })
)

score_summary = (
    score_value
    .merge(score_maximum, on=["阶段", "ai"], how="left")
    .rename(columns={
        "ai": "AI",
        "成果质量": "成果质量分",
        "独立可靠性": "独立可靠性分",
    })
    [
        [
            "阶段", "AI",
            "成果质量分", "质量满分",
            "独立可靠性分", "可靠性满分",
        ]
    ]
    .sort_values(["阶段", "AI"])
    .reset_index(drop=True)
)

score_summary.columns.name = None

# Claude 混合成果仅计成果质量
score_view = score_summary.copy()
for column in ["成果质量分", "质量满分", "独立可靠性分", "可靠性满分"]:
    score_view[column] = score_view[column].map(
        lambda value: "—" if pd.isna(value) else int(value)
    )

score_view

# 对照本次裁定，原评分与修订后结果并列。
import 评测数据处理 as pipeline
current_tables=dict(pipeline.load_core().extract(DB)[0])
current_rows=[]
for run in current_tables['运行记录']:
    ai,rnd=run['AI'],run['轮次']
    mm=[r for r in current_tables['评分与用量'] if r['参与编号']==run['参与编号']]
    qq=[r['数值'] for r in mm if r['指标类别']=='质量分项']
    rr=[r['数值'] for r in mm if r['指标类别']=='可靠性分项']
    cap=sum(r['数值'] for r in mm if r['指标类别']=='质量封顶')
    original=score_summary.loc[(score_summary['AI']==ai)&(score_summary['阶段']==('混合成果' if ai=='Claude' else rnd)),'成果质量分']
    current_rows.append({'AI':ai,'轮次':rnd,'原冻结Q':original.iloc[0] if len(original) else None,'复审分项合计':sum(qq) if qq else None,'封顶调整':cap if qq else None,'最终Q':sum(qq)+cap if qq else None,'R':sum(rr) if rr else None,'参与类别':run['参与类别']})
score_view=pd.DataFrame(current_rows)
score_view


AI,轮次,原冻结Q,复审分项合计,封顶调整,最终Q,R,参与类别
GPT,第一轮,93.0,93.0,0.0,93.0,97.0,第一轮主样本
Grok,第一轮,94.0,94.0,0.0,94.0,99.0,第一轮主样本
DeepSeek,第一轮,89.0,89.0,0.0,89.0,92.0,第一轮主样本
Gemini,第一轮,49.0,49.0,0.0,49.0,67.0,第一轮主样本
Qwen,第一轮,81.0,81.0,0.0,81.0,89.0,第一轮主样本
GLM,第一轮,86.0,86.0,0.0,86.0,94.0,第一轮主样本
DeepSeek,第二轮,95.0,95.0,0.0,95.0,94.0,第二轮主样本
GPT,第二轮,98.0,100.0,0.0,100.0,98.0,第二轮主样本
GLM,第二轮,94.0,94.0,0.0,94.0,96.0,第二轮主样本
Kimi,第二轮,NaN,NaN,NaN,NaN,NaN,补充失败样本


In [5]:
import json

# 展开核后 usage
usage_raw = pd.read_sql_query(
    """
    SELECT run_id, record_json
    FROM autonomy_usage
    WHERE version = 'autonomy_review_20260917_v1'
    ORDER BY run_id
    """,
    con,
)

usage_table = pd.json_normalize(
    usage_raw["record_json"].map(json.loads)
)

usage_view = (
    usage_table
    .assign(轮次=usage_table["轮次"].map({1: "第一轮", 2: "第二轮"}))
    .rename(columns={
        "核后未缓存输入": "未缓存输入",
        "核后缓存读": "缓存读取",
        "核后缓存写": "缓存写入",
        "核后输出": "输出Token",
        "run_id": "运行编号",
    })
    [
        [
            "AI", "轮次", "运行编号",
            "未缓存输入", "缓存读取", "缓存写入", "输出Token",
        ]
    ]
)

usage_view


AI,轮次,运行编号,未缓存输入,缓存读取,缓存写入,输出Token
DeepSeek,第一轮,20260916_153015_DeepSeek_a32f78bc,311371,12838400,0,95925
Qwen,第一轮,20260916_192639_Qwen_8fc12d57,2022,1721958,81580,31090
GLM,第一轮,20260916_210900_GLM_ce9faeda,73925,1867520,0,23039
DeepSeek,第二轮,20260917_103101_DeepSeek_6a24da15,176433,8724224,0,79939
GLM,第二轮,20260917_104919_GLM_b5270231,89382,2327552,0,39647
Kimi,第二轮,20260917_104919_Kimi_006567a6,14461,482816,0,2208
Qwen,第二轮,20260917_104919_Qwen_ca834dcf,3195,8007784,176495,100976


## 4. API计价


In [6]:
import 评测数据处理 as pipeline

# 正式整理结果
core = pipeline.load_core()
tables, evidence, _, _ = core.extract(DB)
analysis_tables = {
    name: pd.DataFrame(rows)
    for name, rows in tables
}

pricing_table = analysis_tables["API计价"].copy()

pricing_summary = (
    pricing_table
    .pivot_table(
        index=["AI", "轮次"],
        columns="计价档",
        values="分项费用（元）",
        aggfunc="sum",
    )
    .reset_index()
    .rename(columns={"低": "低档成本（元）", "高": "高档成本（元）"})
)

pricing_summary.columns.name = None
pricing_summary


AI,轮次,低档成本（元）,高档成本（元）
DeepSeek,第一轮,1.978911,3.957822
DeepSeek,第二轮,1.368611,2.737223
GLM,第一轮,4.971532,4.971532
GLM,第二轮,6.480276,6.480276
GPT,第一轮,24.950214,24.950214
GPT,第二轮,38.958408,38.958408
Kimi,第二轮,1.475652,1.475652
Qwen,第一轮,4.089162,4.089162
Qwen,第二轮,14.328685,14.328685


## 5. 行为与核验


In [7]:
# 第二轮动作复核
action_table = pd.read_sql_query(
    """
    SELECT ai, sequence_no, tool, label, reason
    FROM evaluation_action
    WHERE version = 'enhanced_action_review_20260918_v1'
    ORDER BY ai, sequence_no
    """,
    con,
)

# 按系统汇总动作标签
action_summary = (
    pd.crosstab(action_table["ai"], action_table["label"])
    .rename(columns={
        "contributory": "直接贡献",
        "necessary_exploration": "必要探索",
        "administrative": "管理动作",
        "waste": "确认无收益",
    })
    .reset_index()
    .rename(columns={"ai": "AI"})
)

action_summary


label,AI,管理动作,直接贡献,必要探索,确认无收益
,DeepSeek,3,62,25,1
,GLM,10,26,13,2
,GPT,4,20,10,2
,Gemini,8,13,26,0
,Grok,13,24,14,0
,Kimi,8,0,1,13
,Qwen,24,38,24,0


In [8]:
# 各类核验记录计数
check_summary = pd.read_sql_query(
    """
    SELECT '数值核验' AS 核验类型, COUNT(*) AS 记录数
    FROM evaluation_numeric_check
    UNION ALL
    SELECT '结论评审', COUNT(*)
    FROM evaluation_headline
    UNION ALL
    SELECT '回溯核验', COUNT(*)
    FROM autonomy_delivery_checks
    UNION ALL
    SELECT '修正记录', COUNT(*)
    FROM autonomy_repairs
    """,
    con,
)

check_summary


核验类型,记录数
数值核验,86
结论评审,56
回溯核验,155
修正记录,45


## 6. 生成表


In [9]:
table_summary = pd.DataFrame(
    [
        {
            "表名": name,
            "行数": len(frame),
            "字段数": len(frame.columns),
        }
        for name, frame in analysis_tables.items()
    ]
)

table_summary


表名,行数,字段数
运行记录,15,12
评分与用量,622,11
API计价,72,14
行为记录,815,20
核验记录,294,22


## 7. 对账


In [10]:
# 分析表连接与主工作簿对账
run_index = (
    analysis_tables["运行记录"]
    .set_index("参与编号")["运行编号"]
)

validation_rows = []
workbook_path = ROOT / "交付成果" / "项目2_AI评测分析.xlsx"

for name, frame in analysis_tables.items():
    exported_table = pd.read_excel(
        workbook_path,
        sheet_name=name,
    )
    exported_source = exported_table[frame.columns]

    participant_ok = frame["参与编号"].isin(run_index.index).all()
    evidence_ok = frame["证据编号"].is_unique

    if name == "运行记录":
        run_ok = True
    else:
        expected_run = frame["参与编号"].map(run_index)
        run_ok = (
            frame["运行编号"].fillna("").astype(str)
            == expected_run.fillna("").astype(str)
        ).all()

    same_values = True
    for column in frame.columns:
        left = frame[column]
        right = exported_source[column]

        if pd.api.types.is_numeric_dtype(left):
            left_num = pd.to_numeric(left, errors="coerce")
            right_num = pd.to_numeric(right, errors="coerce")
            same_column = (
                ((left_num - right_num).abs() <= 1e-9)
                | (left_num.isna() & right_num.isna())
            ).all()
        else:
            same_column = (
                left.fillna("").astype(str)
                == right.fillna("").astype(str)
            ).all()

        if not same_column:
            same_values = False
            break

    validation_rows.append({
        "表名": name,
        "行数一致": len(frame) == len(exported_table),
        "字段齐全": set(frame.columns).issubset(exported_table.columns),
        "内容一致": same_values,
        "参与编号有效": participant_ok,
        "运行编号有效": run_ok,
        "证据编号唯一": evidence_ok,
    })

validation_table = pd.DataFrame(validation_rows)
validation_table["通过"] = validation_table.drop(columns="表名").all(axis=1)

assert validation_table["通过"].all()

validation_table

from openpyxl import load_workbook

book = load_workbook(workbook_path, data_only=False)
assert {"评测总览", "单AI分析", "评分细项", "分维度比较"}.issubset(book.sheetnames)
assert len(book["评测总览"]._charts) == 4
assert len(book["单AI分析"]._charts) == 4
assert book["评测总览"]["E3"].value == "▼"
assert book["单AI分析"]["E3"].value == "▼"
assert book["单AI分析"]["I3"].value == "▼"
assert "#REF!" not in str(book["单AI分析"]["H5"].value)
book.close()

assert any(str(v.sqref)=="C3" for v in book["评测总览"].data_validations.dataValidation)
assert {str(v.sqref) for v in book["单AI分析"].data_validations.dataValidation} == {"C3", "G3"}


## 8. 从分数对照到研究结论
固定六个两轮主样本配对；Kimi失败记录与Claude混合交付单独说明。先分解Q增量，再检查单样本影响，最后用问题归属核对修正负担。下列计算只读当前Excel缓存，运行前先在Excel重算保存。


In [11]:
from pathlib import Path
from collections import defaultdict, Counter
from statistics import mean, median
import openpyxl
root = Path(r"D:\项目2")
book = openpyxl.load_workbook(root / "交付成果" / "项目2_AI评测分析.xlsx", data_only=True)
systems = ["GPT", "Grok", "DeepSeek", "GLM", "Qwen"]
runs = [r for r in list(book["运行记录"].values)[1:] if r[0] in systems]
paired = {ai: {r[1]: r for r in runs if r[0] == ai} for ai in systems}
dq = [paired[a]["第二轮"][13] - paired[a]["第一轮"][13] for a in systems]
dr = [paired[a]["第二轮"][14] - paired[a]["第一轮"][14] for a in systems]
print("配对样本", len(paired), "Q增量", dq, "R增量", dr)
print("Q平均/中位增量", mean(dq), median(dq))
print("R平均/中位增量", mean(dr), median(dr))
scores = defaultdict(float)
metrics = list(book["评分与用量"].values)[1:]
for r in metrics:
    if r[0] in systems and r[2] == "质量分项": scores[(r[1], r[3][0])] += r[4]
delta = {k: scores[("第二轮", k)] - scores[("第一轮", k)] for k in "ABCDE"}
cap = sum(r[4] for r in metrics if r[0] in systems and r[1]=="第二轮" and r[2]=="质量封顶")
assert sum(delta.values()) + cap == sum(dq) == 3
assert sum(delta.values()) == 34 and cap == -31
for k in "ABCDE":
    print(k, "一轮均分", scores[("第一轮", k)]/5, "二轮均分", scores[("第二轮", k)]/5,
          "总增分", delta[k], "增量占比", delta[k]/sum(delta.values()))
tasks = [r for r in list(book["核验记录"].values)[1:] if r[0] in systems and r[1] == "第二轮"
         and r[2] == "交付后修正任务" and r[16] == "模型"]
print("模型修正任务", len(tasks), dict(Counter(r[0] for r in tasks)), dict(Counter(r[21] for r in tasks)))
assert len(tasks) == 5
for ai in systems:
    for rnd in ["第一轮", "第二轮"]:
        total = sum(r[4] for r in metrics if r[0]==ai and r[1]==rnd and r[2] in ("质量分项","质量封顶"))
        assert total == paired[ai][rnd][13], (ai,rnd,total)
print("通过：10个Q总分与分项及封顶调整一致；修正任务排除评测方勘误和文件身份记录。")
book.close()


配对样本 5 Q增量 [7, 4, 6, 8, -22] R增量 [1, 0, 2, 2, 6]
Q平均/中位增量 0.6 6
R平均/中位增量 2.2 2
A 一轮均分 18.8 二轮均分 19.6 总增分 4.0 增量占比 0.11764705882352941
B 一轮均分 22.0 二轮均分 23.0 总增分 5.0 增量占比 0.14705882352941177
C 一轮均分 24.0 二轮均分 29.2 总增分 26.0 增量占比 0.7647058823529411
D 一轮均分 14.2 二轮均分 14.0 总增分 -1.0 增量占比 -0.029411764705882353
E 一轮均分 9.6 二轮均分 9.6 总增分 0.0 增量占比 0.0
模型修正任务 5 {'DeepSeek': 1, 'GLM': 1, 'Qwen': 2, 'Grok': 1} {'时间与比较口径': 2, '数值或对象错配': 1, '构成与分类解释': 2}
通过：10个Q总分与分项及封顶调整一致；修正任务排除评测方勘误和文件身份记录。


### 分析解释与归属边界
C调查推进贡献33/61=54.1%的总增分，D最终交付仅贡献2/61=3.3%；因而不把全部提升解释为终检改善。剔除Gemini后平均增量从10.2降至6.4分，方向保持但幅度受单样本影响。分项满分不同，贡献占比用于分解总分，不能当作能力提升率。
Qwen同月重算由评测方完成，不能作为模型自主反证的证据；模型动作依据核验记录中CHK:BEH:Qwen系列。10项模型修正任务中的两项Qwen任务是描述构成和类别首次出现/改名叙述，同月窗口复算另作补充审计，避免把问题一一错配。任务数不等于人工分钟，各任务严重性和成本不同。


## 9. 各AI维度比较
按两轮和AI汇总五个Q维度与五个R维度。Claude为辅助成品Q，Kimi第一轮退出。未评分留空，不混入六系统配对平均。


In [12]:
from pathlib import Path
from collections import defaultdict
import openpyxl
book = openpyxl.load_workbook(Path(r"D:\项目2\交付成果\项目2_AI评测分析.xlsx"), data_only=True)
metrics = list(book["评分与用量"].values)[1:]
group = defaultdict(float)
for r in metrics:
    if r[2] in ("质量分项", "可靠性分项"):
        key = r[3].split()[0]
        group[(r[0], r[1], key if key.startswith("R") else key[0])] += r[4]
for ai in ["GPT", "Grok", "DeepSeek", "GLM", "Qwen", "Gemini", "Claude", "Kimi"]:
    print(ai)
    for rnd in ["第一轮", "第二轮"]:
        q = [group.get((ai,rnd,k)) for k in "ABCDE"]
        reliability = [group.get((ai,rnd,k)) for k in ["R1","R2","R3","R4","R5"]]
        print(rnd, "A-E", q, "R1-R5", reliability)
        if all(v is not None for v in q):
            run = next(r for r in list(book["运行记录"].values)[1:] if r[0]==ai and r[1]==rnd)
            cap=sum(r[4] for r in metrics if r[0]==ai and r[1]==rnd and r[2]=="质量封顶")
            assert sum(q)+cap==run[13], (ai,rnd)
print("通过：分项合计加封顶调整与Q一致，缺失分项保留None，Claude第一轮Q=81。")


GPT
第一轮 A-E [19.0, 25.0, 24.0, 15.0, 10.0] R1-R5 [25.0, 23.0, 19.0, 20.0, 10.0]
第二轮 A-E [20.0, 25.0, 30.0, 15.0, 10.0] R1-R5 [25.0, 24.0, 19.0, 20.0, 10.0]
Grok
第一轮 A-E [20.0, 24.0, 26.0, 14.0, 10.0] R1-R5 [25.0, 24.0, 20.0, 20.0, 10.0]
第二轮 A-E [20.0, 24.0, 30.0, 14.0, 10.0] R1-R5 [25.0, 25.0, 20.0, 19.0, 10.0]
DeepSeek
第一轮 A-E [20.0, 19.0, 28.0, 13.0, 9.0] R1-R5 [25.0, 23.0, 17.0, 17.0, 10.0]
第二轮 A-E [20.0, 22.0, 30.0, 13.0, 10.0] R1-R5 [25.0, 23.0, 19.0, 17.0, 10.0]
GLM
第一轮 A-E [18.0, 23.0, 21.0, 15.0, 9.0] R1-R5 [25.0, 22.0, 19.0, 18.0, 10.0]
第二轮 A-E [19.0, 23.0, 28.0, 14.0, 10.0] R1-R5 [25.0, 23.0, 19.0, 19.0, 10.0]
Qwen
第一轮 A-E [17.0, 19.0, 21.0, 14.0, 10.0] R1-R5 [25.0, 18.0, 19.0, 17.0, 10.0]
第二轮 A-E [19.0, 21.0, 28.0, 14.0, 8.0] R1-R5 [25.0, 22.0, 20.0, 18.0, 10.0]
Gemini
第一轮 A-E [11.0, 10.0, 14.0, 9.0, 5.0] R1-R5 [24.0, 13.0, 12.0, 11.0, 7.0]
第二轮 A-E [19.0, 16.0, 23.0, 12.0, 8.0] R1-R5 [24.0, 16.0, 20.0, 13.0, 8.0]
Claude
第一轮 A-E [17.0, 16.0, 25.0, 14.0, 9.0] R1-R5 [None, None

通过：分项合计加封顶调整与Q一致，缺失分项保留None，Claude第一轮Q=81。
